# Lab 20 — threshold, calibration và HR-ratio ablation

Mục tiêu: tiếp tục từ Lab 19 với hai ứng viên chính và kiểm tra riêng công thức nhịp tim kỳ vọng.

1. `age_bin + LightGBM` — ứng viên ưu tiên recall.
2. `baseline + Logistic Regression` — ứng viên ưu tiên calibration.
3. Ablation: thêm `hr_expected_ratio = thalach / (220 - age)` vào `age_bin + LightGBM`.

Threshold và Platt calibration chỉ được fit từ inner LOCO trên ba training hospitals. Outer test hospital là dữ liệu thật bị khóa.

In [ ]:
!pip -q install lightgbm seaborn

import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score,
    brier_score_loss, confusion_matrix, f1_score, fbeta_score,
    precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
FIXED_THRESHOLD = 0.50
THRESHOLD_GRID = np.arange(0.10, 0.91, 0.01)
OUTPUT_DIR = Path('/content/uci_multicenter_threshold_calibration_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURES = ['age','sex','cp','trestbps','chol','fbs','restecg','thalach',
            'exang','oldpeak','slope','ca','thal']
TARGET = 'target'
NUMERICAL_FEATURES = ['age','trestbps','chol','thalach','oldpeak']
CATEGORICAL_FEATURES = ['sex','cp','fbs','restecg','exang','slope','ca','thal']
RATIO_FEATURES = ['chol_per_age','bps_per_age','hr_ratio']
EXPECTED_HR_FEATURE = 'hr_expected_ratio'
AGE_BIN_FEATURE = 'age_bin'
AGE_BINS = [0, 39, 49, 59, 69, 120]
AGE_LABELS = [0, 1, 2, 3, 4]
BASE_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease'
FILES = {'cleveland':'processed.cleveland.data', 'hungarian':'processed.hungarian.data',
         'switzerland':'processed.switzerland.data', 'va':'processed.va.data'}
COLUMNS = FEATURES + ['num']

def read_uci(site, filename):
    frame = pd.read_csv(f'{BASE_URL}/{filename}', names=COLUMNS, na_values=['?'],
                        skipinitialspace=True).apply(pd.to_numeric, errors='coerce')
    frame[TARGET] = (frame['num'] > 0).astype('int8')
    frame['site'] = site
    return frame[FEATURES + [TARGET, 'site']]

data = pd.concat([read_uci(site, filename) for site, filename in FILES.items()], ignore_index=True)
assert len(data) == 920, f'Expected 920 rows, got {len(data)}'
SITES = list(FILES.keys())
display(data.groupby('site')[TARGET].agg(['size','sum','mean']).round(4))

## 1. Feature sets và model pipeline

`hr_ratio = thalach / age` của Lab 19 không bị xóa; công thức mới được thử riêng dưới tên `hr_expected_ratio`. Cả hai ratio không được đưa cùng lúc vào ablation để tránh khó diễn giải.

In [ ]:
FEATURE_CONFIGS = {
    'F0_P1_baseline': {'age_bin': False, 'ratios': False, 'expected_hr': False},
    'F2_P1_age_bin': {'age_bin': True, 'ratios': False, 'expected_hr': False},
    'F4_P1_age_bin_expected_hr': {'age_bin': True, 'ratios': False, 'expected_hr': True},
}
EXPERIMENTS = {
    'candidate_age_bin_LightGBM': {'feature_config': 'F2_P1_age_bin', 'model': 'LightGBM'},
    'candidate_baseline_Logistic': {'feature_config': 'F0_P1_baseline', 'model': 'Logistic Regression'},
    'ablation_expected_hr_LightGBM': {'feature_config': 'F4_P1_age_bin_expected_hr', 'model': 'LightGBM'},
}

def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors='coerce')
    for column in ['trestbps', 'chol']:
        out.loc[out[column] <= 0, column] = np.nan
    return out

def safe_ratio(numerator, denominator):
    denominator = denominator.where(denominator > 0, np.nan)
    return (numerator / denominator).replace([np.inf, -np.inf], np.nan)

def make_features(frame, feature_config):
    out = apply_p1(frame)
    options = FEATURE_CONFIGS[feature_config]
    if options['ratios']:
        out['chol_per_age'] = safe_ratio(out['chol'], out['age'])
        out['bps_per_age'] = safe_ratio(out['trestbps'], out['age'])
        out['hr_ratio'] = safe_ratio(out['thalach'], out['age'])
    if options['expected_hr']:
        out['hr_expected_ratio'] = safe_ratio(out['thalach'], 220 - out['age'])
    if options['age_bin']:
        out['age_bin'] = pd.cut(out['age'], bins=AGE_BINS, labels=AGE_LABELS,
                                include_lowest=True).astype('object')
    return out

def feature_groups(feature_config):
    options = FEATURE_CONFIGS[feature_config]
    numeric = NUMERICAL_FEATURES.copy()
    categorical = CATEGORICAL_FEATURES.copy()
    if options['ratios']:
        numeric += RATIO_FEATURES
    if options['expected_hr']:
        numeric += [EXPECTED_HR_FEATURE]
    if options['age_bin']:
        categorical += [AGE_BIN_FEATURE]
    return numeric, categorical

def build_pipeline(feature_config, model_name):
    numeric_features, categorical_features = feature_groups(feature_config)
    numeric = Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)),
                       ('scaler', StandardScaler())])
    categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
                            ('encoder', OneHotEncoder(handle_unknown='ignore'))])
    preprocessor = ColumnTransformer([('numeric', numeric, numeric_features),
                                     ('categorical', categorical, categorical_features)])
    if model_name == 'Logistic Regression':
        estimator = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    else:
        estimator = LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15,
            min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
            random_state=RANDOM_STATE, verbosity=-1)
    return Pipeline([('preprocessor', preprocessor), ('classifier', estimator)])

display(pd.DataFrame({name: pd.Series(feature_groups(spec['feature_config'])[0] +
                                   feature_groups(spec['feature_config'])[1])
                      for name, spec in EXPERIMENTS.items()}).T)

## 2. Inner LOCO threshold và Platt calibration

Với mỗi outer test hospital, ba hospital còn lại được dùng cho `GroupKFold(n_splits=3)`. OOF probabilities từ inner folds dùng để chọn threshold tối đa F2 và fit sigmoid calibrator. Không dùng outer test hospital ở bất kỳ bước chọn threshold/calibration nào.

In [ ]:
def safe_roc_auc(y_true, probability):
    return roc_auc_score(y_true, probability) if len(np.unique(y_true)) > 1 else np.nan

def logit_probability(probability):
    clipped = np.clip(np.asarray(probability, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped)).reshape(-1, 1)

def fit_platt_calibrator(probability, y_true):
    calibrator = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000, random_state=RANDOM_STATE)
    calibrator.fit(logit_probability(probability), y_true)
    return calibrator

def apply_platt_calibrator(calibrator, probability):
    return calibrator.predict_proba(logit_probability(probability))[:, 1]

def choose_f2_threshold(y_true, probability):
    scores = []
    for threshold in THRESHOLD_GRID:
        prediction = (probability >= threshold).astype(int)
        scores.append(fbeta_score(y_true, prediction, beta=2, zero_division=0))
    best_score = max(scores)
    candidates = [t for t, score in zip(THRESHOLD_GRID, scores) if np.isclose(score, best_score)]
    return float(min(candidates)), float(best_score)

def inner_oof(train_frame, feature_config, model_name):
    groups = train_frame['site'].to_numpy()
    y = train_frame[TARGET].to_numpy()
    probabilities = np.full(len(train_frame), np.nan, dtype=float)
    splitter = GroupKFold(n_splits=3)
    for inner_train_idx, inner_valid_idx in splitter.split(train_frame, y, groups):
        inner_train = train_frame.iloc[inner_train_idx]
        inner_valid = train_frame.iloc[inner_valid_idx]
        train_ready = make_features(inner_train, feature_config)
        valid_ready = make_features(inner_valid, feature_config)
        numeric_features, categorical_features = feature_groups(feature_config)
        columns = numeric_features + categorical_features
        model = build_pipeline(feature_config, model_name)
        model.fit(train_ready[columns], inner_train[TARGET])
        probabilities[inner_valid_idx] = model.predict_proba(valid_ready[columns])[:, 1]
    assert np.isfinite(probabilities).all(), 'Inner OOF probabilities are incomplete'
    return y, probabilities

def expected_calibration_error(y_true, probability, n_bins=10):
    y_true = np.asarray(y_true); probability = np.asarray(probability)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probability >= left) & (probability <= right if right == 1 else probability < right)
        if mask.any():
            ece += mask.mean() * abs(probability[mask].mean() - y_true[mask].mean())
    return float(ece)

def score_predictions(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {'accuracy': accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'pr_auc': average_precision_score(y_true, probability),
        'specificity': tn / (tn + fp) if (tn + fp) else np.nan,
        'f1': f1_score(y_true, prediction, zero_division=0),
        'roc_auc': safe_roc_auc(y_true, probability),
        'brier': brier_score_loss(y_true, probability),
        'ece': expected_calibration_error(y_true, probability),
        'false_negatives': int(fn), 'threshold': float(threshold)}


In [ ]:
results = []
threshold_audits = []
for test_site in SITES:
    train_frame = data[data['site'] != test_site].reset_index(drop=True)
    test_frame = data[data['site'] == test_site].reset_index(drop=True)
    for experiment, spec in EXPERIMENTS.items():
        feature_config, model_name = spec['feature_config'], spec['model']
        numeric_features, categorical_features = feature_groups(feature_config)
        columns = numeric_features + categorical_features
        y_oof, raw_oof = inner_oof(train_frame, feature_config, model_name)
        raw_threshold, raw_f2 = choose_f2_threshold(y_oof, raw_oof)
        calibrator = fit_platt_calibrator(raw_oof, y_oof)
        calibrated_oof = apply_platt_calibrator(calibrator, raw_oof)
        calibrated_threshold, calibrated_f2 = choose_f2_threshold(y_oof, calibrated_oof)
        final_train = make_features(train_frame, feature_config)
        final_test = make_features(test_frame, feature_config)
        final_model = build_pipeline(feature_config, model_name)
        started = time.perf_counter()
        final_model.fit(final_train[columns], train_frame[TARGET])
        fit_seconds = time.perf_counter() - started
        raw_test = final_model.predict_proba(final_test[columns])[:, 1]
        calibrated_test = apply_platt_calibrator(calibrator, raw_test)
        evaluations = [
            ('raw_fixed_0.50', raw_test, FIXED_THRESHOLD),
            ('raw_tuned_F2', raw_test, raw_threshold),
            ('sigmoid_calibrated_fixed_0.50', calibrated_test, FIXED_THRESHOLD),
            ('sigmoid_calibrated_tuned_F2', calibrated_test, calibrated_threshold),
        ]
        threshold_audits.append({'test_site': test_site, 'experiment': experiment,
            'model': model_name, 'feature_config': feature_config,
            'inner_oof_brier_raw': brier_score_loss(y_oof, raw_oof),
            'inner_oof_brier_calibrated': brier_score_loss(y_oof, calibrated_oof),
            'inner_oof_ece_raw': expected_calibration_error(y_oof, raw_oof),
            'inner_oof_ece_calibrated': expected_calibration_error(y_oof, calibrated_oof),
            'raw_threshold': raw_threshold, 'raw_oof_f2': raw_f2,
            'calibrated_threshold': calibrated_threshold, 'calibrated_oof_f2': calibrated_f2})
        for evaluation, probability, threshold in evaluations:
            metrics = score_predictions(test_frame[TARGET].to_numpy(), probability, threshold)
            results.append({'test_site': test_site, 'experiment': experiment,
                'model': model_name, 'feature_config': feature_config,
                'evaluation': evaluation, 'test_rows': len(test_frame),
                'fit_seconds': fit_seconds, **metrics})
    print('Completed outer test site:', test_site)

results_df = pd.DataFrame(results)
threshold_df = pd.DataFrame(threshold_audits)
display(results_df.head())
display(threshold_df.round(4))

## 3. Tổng hợp và xuất kết quả

Đánh giá chính cần xem: recall, false negatives, Brier và ECE. AUC dùng để kiểm tra ranking. Threshold được chọn trên inner OOF, không phải outer test.

In [ ]:
summary = results_df.groupby(['experiment', 'model', 'evaluation']).agg(
    folds=('test_site', 'nunique'), roc_auc_mean=('roc_auc', 'mean'),
    roc_auc_std=('roc_auc', 'std'), roc_auc_worst=('roc_auc', 'min'),
    pr_auc_mean=('pr_auc', 'mean'), recall_mean=('recall', 'mean'),
    recall_std=('recall', 'std'), recall_worst=('recall', 'min'),
    specificity_mean=('specificity', 'mean'), f1_mean=('f1', 'mean'),
    brier_mean=('brier', 'mean'), ece_mean=('ece', 'mean'),
    false_negatives_mean_per_fold=('false_negatives', 'mean'),
    false_negatives_total=('false_negatives', 'sum'), threshold_mean=('threshold', 'mean'),
    fit_seconds_mean=('fit_seconds', 'mean')).reset_index()

display(summary.sort_values(['experiment', 'recall_mean'], ascending=[True, False]).round(6))

plot_data = summary.copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=plot_data, x='experiment', y='recall_mean', hue='evaluation', ax=axes[0])
axes[0].set_title('Mean recall after threshold/calibration'); axes[0].tick_params(axis='x', rotation=25)
sns.barplot(data=plot_data, x='experiment', y='brier_mean', hue='evaluation', ax=axes[1])
axes[1].set_title('Mean Brier score'); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'threshold_calibration_summary.png', dpi=180, bbox_inches='tight'); plt.show()

results_path = OUTPUT_DIR / 'threshold_calibration_loco_results.csv'
threshold_path = OUTPUT_DIR / 'threshold_calibration_inner_oof_audit.csv'
summary_path = OUTPUT_DIR / 'threshold_calibration_summary.csv'
results_df.to_csv(results_path, index=False)
threshold_df.to_csv(threshold_path, index=False)
summary.to_csv(summary_path, index=False)
run_config = {'dataset_rows': 920, 'validation': 'outer LOCO + inner GroupKFold by hospital',
    'preprocessing': 'P1_sentinel_aware', 'fixed_threshold': FIXED_THRESHOLD,
    'threshold_objective': 'max F2 on inner OOF', 'threshold_grid': [0.10, 0.90, 0.01],
    'calibration': 'Platt sigmoid fitted on inner OOF probabilities',
    'age_bins': AGE_BINS, 'experiments': EXPERIMENTS,
    'outer_test_policy': 'real held-out hospital only'}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('/content/uci_multicenter_threshold_calibration_results', 'zip', OUTPUT_DIR)
print('Saved:', OUTPUT_DIR, zip_path)